Notebook 1 — 01_Data_Audit.ipynb
The first notebook will audit the dataset and existing experiment files before we change any models. Create a new Jupyter notebook with this name.

In [6]:

from pathlib import Path

# Your notebook is inside "Notebooks 2", so go to the project root.
PROJECT_ROOT = Path.cwd().parent

print("Project root:", PROJECT_ROOT)
print("\nPotential metadata and feature files:")

extensions = {".csv", ".parquet", ".xlsx", ".json"}

for path in sorted(PROJECT_ROOT.rglob("*")):
    if not path.is_file():
        continue

    if any(part in {".git", ".venv", "venv", "node_modules",
                    "__pycache__"} for part in path.parts):
        continue

    if path.suffix.lower() in extensions:
        print(path.relative_to(PROJECT_ROOT))


Project root: /Users/vs/Sem 5/Respiratory-Disease-Classification

Potential metadata and feature files:
.vscode/settings.json
app/frontend/.prettierrc.json
app/frontend/package-lock.json
app/frontend/package.json
app/frontend/tsconfig.json
app/frontend/vercel.json
data/interim/.ipynb_checkpoints/audio_info-checkpoint.csv
data/interim/.ipynb_checkpoints/cycles-checkpoint.csv
data/interim/.ipynb_checkpoints/cycles_validation-checkpoint.csv
data/interim/.ipynb_checkpoints/filename_validation-checkpoint.csv
data/interim/.ipynb_checkpoints/patients-checkpoint.csv
data/interim/.ipynb_checkpoints/recordings-checkpoint.csv
data/interim/audio_info.csv
data/interim/cycles.csv
data/interim/cycles_validation.csv
data/interim/filename_validation.csv
data/interim/patients.csv
data/interim/recordings.csv
data/processed/.ipynb_checkpoints/audio_info_clean-checkpoint.csv
data/processed/analysis/.ipynb_checkpoints/sound_feature_means-checkpoint.csv
data/processed/analysis/crackle_feature_correlations.cs

In [8]:

from pathlib import Path

# Find the project root even if this notebook is nested in a subfolder.
current = Path.cwd().resolve()

PROJECT_ROOT = next(
    (p for p in [current, *current.parents]
     if (p / "data").is_dir()
     and ((p / "notebooks").is_dir()
          or (p / "models").is_dir())),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root. "
        "Set PROJECT_ROOT manually to your repository folder."
    )

print("Project root:", PROJECT_ROOT)

skip_dirs = {
    ".git", ".venv", "venv", "node_modules",
    "__pycache__", ".ipynb_checkpoints"
}
extensions = {".csv", ".parquet", ".xlsx", ".json"}

files = []
for path in PROJECT_ROOT.rglob("*"):
    if not path.is_file():
        continue
    if any(part in skip_dirs for part in path.parts):
        continue
    if path.suffix.lower() in extensions:
        files.append(path)

print(f"\nFound {len(files)} data/config files:\n")
for i, path in enumerate(sorted(files), start=1):
    print(f"{i:>3}. {path.relative_to(PROJECT_ROOT)}")


Project root: /Users/vs/Sem 5/Respiratory-Disease-Classification

Found 179 data/config files:

  1. .vscode/settings.json
  2. app/frontend/.prettierrc.json
  3. app/frontend/package-lock.json
  4. app/frontend/package.json
  5. app/frontend/tsconfig.json
  6. app/frontend/vercel.json
  7. data/interim/audio_info.csv
  8. data/interim/cycles.csv
  9. data/interim/cycles_validation.csv
 10. data/interim/filename_validation.csv
 11. data/interim/patients.csv
 12. data/interim/recordings.csv
 13. data/processed/analysis/crackle_feature_correlations.csv
 14. data/processed/analysis/crackle_feature_means.csv
 15. data/processed/analysis/disease_feature_means.csv
 16. data/processed/analysis/feature_coefficient_variation.csv
 17. data/processed/analysis/feature_statistics.csv
 18. data/processed/analysis/feature_variance.csv
 19. data/processed/analysis/high_correlation_pairs.csv
 20. data/processed/analysis/sound_feature_means.csv
 21. data/processed/analysis/very_high_correlation_pairs.cs

In [9]:

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()

# Find the repository root automatically.
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (
        (PROJECT_ROOT / "data" / "processed").is_dir()
        and (PROJECT_ROOT / "models").is_dir()
    ):
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "data" / "processed").is_dir():
    raise FileNotFoundError("Could not locate the project root.")

files_to_inspect = [
    "data/interim/recordings.csv",
    "data/interim/cycles.csv",
    "data/interim/patients.csv",
    "data/processed/recordings_clean.csv",
    "data/processed/cycles_clean.csv",
    "data/processed/features/respiratory_cycle_features.csv",
    "data/raw/ICBHI/metadata/diagnosis.csv",
    "data/raw/ICBHI/metadata/demographic.csv",
]

print("Project root:", PROJECT_ROOT)

for relative_path in files_to_inspect:
    path = PROJECT_ROOT / relative_path

    print("\n" + "=" * 75)
    print(relative_path)

    if not path.exists():
        print("File not found — skipped")
        continue

    df = pd.read_csv(path)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head(3))


Project root: /Users/vs/Sem 5/Respiratory-Disease-Classification

data/interim/recordings.csv
Shape: (920, 7)
Columns: ['patient_id', 'recording_id', 'chest_location', 'acquisition_mode', 'equipment', 'wav_path', 'official_split']


,patient_id,recording_id,chest_location,acquisition_mode,equipment,wav_path,official_split
0,101,1b1,Al,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test
1,101,1b1,Pr,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test
2,102,1b1,Ar,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test



data/interim/cycles.csv
Shape: (6898, 12)
Columns: ['patient_id', 'recording_id', 'chest_location', 'acquisition_mode', 'equipment', 'wav_path', 'cycle_number', 'start_time', 'end_time', 'crackles', 'wheezes', 'sound_label']


,patient_id,recording_id,chest_location,acquisition_mode,equipment,wav_path,cycle_number,start_time,end_time,crackles,wheezes,sound_label
0,101,1b1,Al,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,1,0.036,0.579,0,0,Normal
1,101,1b1,Al,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,2,0.579,2.450,0,0,Normal
2,101,1b1,Al,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,3,2.450,3.893,0,0,Normal



data/interim/patients.csv
Shape: (126, 7)
Columns: ['patient_id', 'diagnosis', 'age', 'sex', 'adult_bmi', 'child_weight', 'child_height']


...

,patient_id,recording_id,chest_location,acquisition_mode,equipment,cycle_number,start_time,end_time,sound_label,crackles,...,mfcc_7_mean,mfcc_8_mean,mfcc_9_mean,mfcc_10_mean,mfcc_11_mean,mfcc_12_mean,mfcc_13_mean,wavelet_energy,wavelet_entropy,wavelet_std
0,101,1b1,Al,sc,Meditron,1,0.036,0.579,Normal,0,...,14.649715,13.271879,15.322943,4.408390,9.810836,11.595978,6.332477,110.760926,0.062777,0.224449
1,101,1b1,Al,sc,Meditron,2,0.579,2.450,Normal,0,...,11.704497,17.665747,14.362599,8.428231,8.835912,10.312641,6.039036,263.467394,0.090006,0.187299
2,101,1b1,Al,sc,Meditron,3,2.450,3.893,Normal,0,...,9.776377,21.063862,10.441285,5.727220,12.893122,8.313774,3.833181,169.927915,0.150584,0.171195



data/raw/ICBHI/metadata/diagnosis.csv
Shape: (126, 2)
Columns: ['patient_id', 'diagnosis']


,patient_id,diagnosis
0,101,URTI
1,102,Healthy
2,103,Asthma



data/raw/ICBHI/metadata/demographic.csv
Shape: (126, 7)
Columns: ['patient_id', 'diagnosis', 'age', 'sex', 'adult_bmi', 'child_weight', 'child_height']


,patient_id,diagnosis,age,sex,adult_bmi,child_weight,child_height
0,101,URTI,3.00,F,NaN,19.0,99.0
1,102,Healthy,0.75,F,NaN,9.8,73.0
2,103,Asthma,70.00,F,33.0,NaN,NaN


In [10]:

import pandas as pd
import numpy as np

# Load the real files discovered in your project
patients = pd.read_csv(
    PROJECT_ROOT / "data" / "interim" / "patients.csv"
)

recordings = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "recordings_clean.csv"
)

cycles = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "cycles_clean.csv"
)

features = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "features"
    / "respiratory_cycle_features.csv"
)

print("PATIENTS:", patients.shape)
print("RECORDINGS:", recordings.shape)
print("CYCLES:", cycles.shape)
print("FEATURES:", features.shape)

print("\nMissing diagnosis values:")
print("Patients:", patients["diagnosis"].isna().sum())
print("Features:", features["diagnosis"].isna().sum())

# Each patient should have one diagnosis.
diagnosis_counts = patients.groupby("patient_id")["diagnosis"].nunique()
print("\nPatients with multiple diagnoses:", (diagnosis_counts > 1).sum())

# Check whether all feature rows map to known patients.
unknown_patients = set(features["patient_id"]) - set(patients["patient_id"])
print("Feature rows with unknown patient IDs:", len(unknown_patients))

# Patient-level disease distribution
patient_counts = (
    patients.groupby("diagnosis")["patient_id"]
    .nunique()
    .sort_values(ascending=False)
    .rename("patients")
    .to_frame()
)

patient_counts["percentage"] = (
    patient_counts["patients"] / patient_counts["patients"].sum() * 100
).round(2)

print("\nPATIENT-LEVEL CLASS DISTRIBUTION")
display(patient_counts)

# Cycle-level disease distribution
cycle_counts = (
    features.groupby("diagnosis")
    .size()
    .sort_values(ascending=False)
    .rename("cycles")
    .to_frame()
)

cycle_counts["percentage"] = (
    cycle_counts["cycles"] / cycle_counts["cycles"].sum() * 100
).round(2)

print("\nCYCLE-LEVEL CLASS DISTRIBUTION")
display(cycle_counts)


PATIENTS: (126, 7)
RECORDINGS: (920, 7)
CYCLES: (6898, 12)
FEATURES: (6898, 52)

Missing diagnosis values:
Patients: 0
Features: 0

Patients with multiple diagnoses: 0
Feature rows with unknown patient IDs: 0

PATIENT-LEVEL CLASS DISTRIBUTION


,patients,percentage
diagnosis,,
COPD,64,50.79
Healthy,26,20.63
URTI,14,11.11
Bronchiectasis,7,5.56
Bronchiolitis,6,4.76
Pneumonia,6,4.76
LRTI,2,1.59
Asthma,1,0.79



CYCLE-LEVEL CLASS DISTRIBUTION


,cycles,percentage
diagnosis,,
COPD,5746,83.30
Healthy,322,4.67
Pneumonia,285,4.13
URTI,243,3.52
Bronchiolitis,160,2.32
Bronchiectasis,104,1.51
LRTI,32,0.46
Asthma,6,0.09


In [11]:

patient_class_counts = patients.groupby("diagnosis")["patient_id"].nunique()

imbalance_report = pd.DataFrame({
    "patients": patient_class_counts,
    "cycles": features.groupby("diagnosis").size()
})

imbalance_report["cycles_per_patient"] = (
    imbalance_report["cycles"] / imbalance_report["patients"]
).round(2)

imbalance_report = imbalance_report.sort_values("patients")

display(imbalance_report)

print("\nMinimum patients in any class:", patient_class_counts.min())
print("Number of diagnosis classes:", patients["diagnosis"].nunique())

print("\nClasses with fewer than 5 patients:")
display(imbalance_report[imbalance_report["patients"] < 5])


,patients,cycles,cycles_per_patient
diagnosis,,,
Asthma,1,6,6.00
LRTI,2,32,16.00
Bronchiolitis,6,160,26.67
Pneumonia,6,285,47.50
Bronchiectasis,7,104,14.86
URTI,14,243,17.36
Healthy,26,322,12.38
COPD,64,5746,89.78



Minimum patients in any class: 1
Number of diagnosis classes: 8

Classes with fewer than 5 patients:


,patients,cycles,cycles_per_patient
diagnosis,,,
Asthma,1,6,6.0
LRTI,2,32,16.0


In [12]:

import json
import pandas as pd

manifest_paths = {
    "train": PROJECT_ROOT / "data/processed/deep_learning/train_manifest.csv",
    "validation": PROJECT_ROOT / "data/processed/deep_learning/validation_cnn_manifest.csv",
    "test": PROJECT_ROOT / "data/processed/deep_learning/test_manifest.csv",
}

manifests = {}

for split_name, path in manifest_paths.items():
    print(f"\n{'=' * 65}\n{split_name.upper()}")

    if not path.exists():
        print("Manifest not found:", path)
        continue

    df = pd.read_csv(path)
    manifests[split_name] = df

    print("File:", path.relative_to(PROJECT_ROOT))
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head(3))

print("\nPatient-ID overlap checks")

patient_column_candidates = [
    "patient_id", "patient", "patientId", "patient_ID"
]

split_patient_sets = {}

for split_name, df in manifests.items():
    patient_col = next(
        (col for col in patient_column_candidates if col in df.columns),
        None
    )

    if patient_col is None:
        print(
            f"{split_name}: patient ID column not identified automatically. "
            "Review the displayed columns."
        )
        continue

    ids = set(df[patient_col].dropna().astype(str))
    split_patient_sets[split_name] = ids
    print(f"{split_name}: {len(ids)} distinct patients")

split_names = list(split_patient_sets)

for i, first in enumerate(split_names):
    for second in split_names[i + 1:]:
        overlap = split_patient_sets[first] & split_patient_sets[second]
        print(
            f"{first} ∩ {second}: {len(overlap)} overlapping patients"
        )
        if overlap:
            print("Example overlapping IDs:", sorted(overlap)[:10])

protocol_path = (
    PROJECT_ROOT / "reports/research_audit/common_patient_split/protocol.json"
)

if protocol_path.exists():
    print("\nExisting research-audit protocol:")
    with open(protocol_path, "r", encoding="utf-8") as f:
        protocol = json.load(f)
    print(json.dumps(protocol, indent=2))
else:
    print("\nResearch-audit protocol file not found.")



TRAIN
File: data/processed/deep_learning/train_manifest.csv
Shape: (4142, 7)
Columns: ['patient_id', 'recording_id', 'chest_location', 'diagnosis', 'disease_label', 'official_split', 'audio_path']


,patient_id,recording_id,chest_location,diagnosis,disease_label,official_split,audio_path
0,103,2b2,Ar,Asthma,0,train,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
1,103,2b2,Ar,Asthma,0,train,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
2,103,2b2,Ar,Asthma,0,train,/Users/vs/Sem 5/Respiratory-Disease-Classifica...



VALIDATION
Manifest not found: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/validation_cnn_manifest.csv

TEST
File: data/processed/deep_learning/test_manifest.csv
Shape: (2756, 7)
Columns: ['patient_id', 'recording_id', 'chest_location', 'diagnosis', 'disease_label', 'official_split', 'audio_path']


,patient_id,recording_id,chest_location,diagnosis,disease_label,official_split,audio_path
0,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
1,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
2,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...



Patient-ID overlap checks
train: 79 distinct patients
test: 49 distinct patients
train ∩ test: 2 overlapping patients
Example overlapping IDs: ['156', '218']

Existing research-audit protocol:
{
  "protocol": "cycle-level evaluation on the existing CNN patient-disjoint holdout",
  "train_patients": [
    102,
    103,
    104,
    105,
    108,
    111,
    112,
    113,
    114,
    115,
    117,
    118,
    120,
    124,
    126,
    127,
    129,
    130,
    131,
    132,
    133,
    134,
    136,
    137,
    138,
    139,
    140,
    142,
    143,
    144,
    146,
    148,
    149,
    151,
    153,
    154,
    155,
    156,
    160,
    162,
    163,
    164,
    167,
    168,
    170,
    171,
    173,
    174,
    176,
    178,
    180,
    181,
    182,
    184,
    186,
    187,
    188,
    191,
    192,
    194,
    195,
    197,
    198,
    199,
    200,
    201,
    202,
    204,
    207,
    208,
    209,
    210,
    212,
    213,
    214,
    215,
    216,
    

In [13]:

from pathlib import Path
import pandas as pd
import json

manifest_candidates = [
    PROJECT_ROOT / "data/processed/deep_learning",
    PROJECT_ROOT / "data/processed/deep_learning/cnn_baseline",
    PROJECT_ROOT / "reports/research_audit/common_patient_split",
]

manifest_files = []

for folder in manifest_candidates:
    if folder.exists():
        manifest_files.extend(
            p for p in folder.glob("*manifest*.csv")
            if p.is_file()
        )

# Also inspect notebook files to identify the intended split construction.
notebook_root = PROJECT_ROOT / "notebooks"
if notebook_root.exists():
    print("Notebook files:")
    for p in sorted(notebook_root.glob("*.ipynb")):
        print(" ", p.name)

print("\nMANIFEST AUDIT")
for path in sorted(set(manifest_files)):
    df = pd.read_csv(path)

    patient_col = next(
        (c for c in ["patient_id", "patient", "patientId"]
         if c in df.columns),
        None
    )
    diagnosis_col = next(
        (c for c in ["diagnosis", "disease_label", "label"]
         if c in df.columns),
        None
    )

    print("\n" + "-" * 75)
    print("File:", path.relative_to(PROJECT_ROOT))
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    if patient_col:
        print("Unique patients:", df[patient_col].nunique())
    if diagnosis_col:
        print("Disease distribution:")
        display(df[diagnosis_col].value_counts(dropna=False).to_frame("rows"))

# Compare current manifests with the canonical protocol.
protocol_path = (
    PROJECT_ROOT / "reports/research_audit/common_patient_split/protocol.json"
)

if protocol_path.exists():
    with open(protocol_path, "r", encoding="utf-8") as f:
        protocol = json.load(f)

    print("\nPROTOCOL COUNTS")
    for key in ["train_cycles", "validation_cycles", "test_cycles",
                "test_recordings", "feature_count"]:
        print(f"{key}: {protocol.get(key)}")

    print("\nPROTOCOL PATIENT COUNTS")
    for key in ["train_patients",
                "validation_patients_not_used_for_classical_fit",
                "test_patients"]:
        values = protocol.get(key, [])
        print(f"{key}: {len(values)} patients")


Notebook files:
  01_dataset_understanding.ipynb
  02_data_cleaning.ipynb
  03_eda.ipynb
  04_audio_preprocessing.ipynb
  05_feature_engineering.ipynb
  06_feature_analysis.ipynb
  07_feature_selection.ipynb
  08_pca_and_clustering.ipynb
  09_classical_ml.ipynb
  10_deep_learning_data_preparation.ipynb
  11_cnn_baseline.ipynb
  12_multifeature_cnn.ipynb
  13_cnn_lstm_optional.ipynb
  14_patient_level_prediction.ipynb
  15_xai.ipynb
  16_final_model_comparison.ipynb
  17_Final_Artifact_Generation_and_Validation.ipynb
  18_model_validation.ipynb

MANIFEST AUDIT

---------------------------------------------------------------------------
File: data/processed/deep_learning/cnn_baseline/cnn_patient_disjoint_manifest.csv
Rows: 6898
Columns: ['patient_id', 'recording_id', 'chest_location', 'diagnosis', 'disease_label', 'official_split', 'audio_path', 'cnn_split']
Unique patients: 126
Disease distribution:


,rows
diagnosis,
COPD,5746
Healthy,322
Pneumonia,285
URTI,243
Bronchiolitis,160
Bronchiectasis,104
LRTI,32
Asthma,6



---------------------------------------------------------------------------
File: data/processed/deep_learning/cnn_baseline/test_cnn_manifest.csv
Rows: 1561
Columns: ['patient_id', 'recording_id', 'chest_location', 'diagnosis', 'disease_label', 'official_split', 'audio_path', 'cnn_split']
Unique patients: 25
Disease distribution:


,rows
diagnosis,
COPD,1337
Healthy,84
Pneumonia,52
URTI,49
Bronchiectasis,20
Bronchiolitis,19



---------------------------------------------------------------------------
File: data/processed/deep_learning/cnn_baseline/train_cnn_manifest.csv
Rows: 4423
Columns: ['patient_id', 'recording_id', 'chest_location', 'diagnosis', 'disease_label', 'official_split', 'audio_path', 'cnn_split']
Unique patients: 82
Disease distribution:


...


---------------------------------------------------------------------------
File: data/processed/deep_learning/test_manifest.csv
Rows: 2756
Columns: ['patient_id', 'recording_id', 'chest_location', 'diagnosis', 'disease_label', 'official_split', 'audio_path']
Unique patients: 49
Disease distribution:


,rows
diagnosis,
COPD,2409
Healthy,175
Bronchiolitis,86
URTI,72
Bronchiectasis,14



---------------------------------------------------------------------------
File: data/processed/deep_learning/train_manifest.csv
Rows: 4142
Columns: ['patient_id', 'recording_id', 'chest_location', 'diagnosis', 'disease_label', 'official_split', 'audio_path']
Unique patients: 79
Disease distribution:


,rows
diagnosis,
COPD,3337
Pneumonia,285
URTI,171
Healthy,147
Bronchiectasis,90
Bronchiolitis,74
LRTI,32
Asthma,6



PROTOCOL COUNTS
train_cycles: 4423
validation_cycles: 914
test_cycles: 1561
test_recordings: 53
feature_count: 33

PROTOCOL PATIENT COUNTS
train_patients: 82 patients
validation_patients_not_used_for_classical_fit: 19 patients
test_patients: 25 patients


In [14]:

import pandas as pd

split_dir = PROJECT_ROOT / "data/processed/deep_learning/cnn_baseline"

split_paths = {
    "train": split_dir / "train_cnn_manifest.csv",
    "validation": split_dir / "validation_cnn_manifest.csv",
    "test": split_dir / "test_cnn_manifest.csv",
}

splits = {
    name: pd.read_csv(path)
    for name, path in split_paths.items()
}

required_columns = {
    "patient_id", "recording_id", "chest_location",
    "diagnosis", "audio_path"
}

for name, df in splits.items():
    missing = required_columns - set(df.columns)
    assert not missing, f"{name}: missing columns {missing}"

    print(
        f"{name}: {len(df)} rows, "
        f"{df['patient_id'].nunique()} patients, "
        f"{df['diagnosis'].nunique()} represented diagnoses"
    )

# 1. Check patient overlap across splits.
patient_sets = {
    name: set(df["patient_id"].dropna())
    for name, df in splits.items()
}

print("\nPATIENT OVERLAP")
for first, second in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test"),
]:
    overlap = patient_sets[first] & patient_sets[second]
    print(f"{first} vs {second}: {len(overlap)} overlapping patients")
    if overlap:
        print("Overlapping IDs:", sorted(overlap))

# 2. Check whether cycle-level rows are duplicated across splits.
# A cycle is identified using its patient, recording, chest location,
# and cycle timing when timing columns are available.
cycle_key = [
    "patient_id", "recording_id", "chest_location",
    "start_time", "end_time"
]

if all(c in splits["train"].columns for c in cycle_key):
    split_keys = {}
    for name, df in splits.items():
        split_keys[name] = set(
            map(tuple, df[cycle_key].itertuples(index=False, name=None))
        )

    print("\nCYCLE-KEY OVERLAP")
    for first, second in [
        ("train", "validation"),
        ("train", "test"),
        ("validation", "test"),
    ]:
        print(
            f"{first} vs {second}: "
            f"{len(split_keys[first] & split_keys[second])} shared cycle keys"
        )
else:
    print(
        "\nCycle timing columns are unavailable in these manifests; "
        "cycle-key overlap requires a separate recording/cycle mapping."
    )

# 3. Verify counts and disease support.
print("\nSPLIT SUMMARY")
summary_rows = []

for name, df in splits.items():
    summary_rows.append({
        "split": name,
        "rows": len(df),
        "unique_patients": df["patient_id"].nunique(),
        "unique_recordings": df[
            ["patient_id", "recording_id"]
        ].drop_duplicates().shape[0],
    })

display(pd.DataFrame(summary_rows))

print("\nDISEASE SUPPORT BY SPLIT")
support = pd.DataFrame({
    name: df["diagnosis"].value_counts()
    for name, df in splits.items()
}).fillna(0).astype(int)

display(support)

print("\nTotal rows:", sum(len(df) for df in splits.values()))
print("Total distinct patients:", len(set.union(*patient_sets.values())))


train: 4423 rows, 82 patients, 8 represented diagnoses
validation: 914 rows, 19 patients, 6 represented diagnoses
test: 1561 rows, 25 patients, 6 represented diagnoses

PATIENT OVERLAP
train vs validation: 0 overlapping patients
train vs test: 0 overlapping patients
validation vs test: 0 overlapping patients

Cycle timing columns are unavailable in these manifests; cycle-key overlap requires a separate recording/cycle mapping.

SPLIT SUMMARY


,split,rows,unique_patients,unique_recordings
0,train,4423,82,166
1,validation,914,19,37
2,test,1561,25,53



DISEASE SUPPORT BY SPLIT


,train,validation,test
diagnosis,,,
Asthma,6,0,0
Bronchiectasis,76,8,20
Bronchiolitis,114,27,19
COPD,3713,696,1337
Healthy,194,44,84
LRTI,32,0,0
Pneumonia,121,112,52
URTI,167,27,49



Total rows: 6898
Total distinct patients: 126


In [15]:

import pandas as pd

cycles_path = PROJECT_ROOT / "data/processed/cycles_clean.csv"
features_path = (
    PROJECT_ROOT / "data/processed/features/respiratory_cycle_features.csv"
)

cycles = pd.read_csv(cycles_path)
features = pd.read_csv(features_path)

# 1. Check consistency between patient labels and cycle feature labels.
patient_labels = patients[["patient_id", "diagnosis"]].drop_duplicates()

feature_labels = features[["patient_id", "diagnosis"]].drop_duplicates()
label_check = feature_labels.merge(
    patient_labels,
    on="patient_id",
    how="left",
    suffixes=("_feature", "_patient"),
    indicator=True
)

label_check["label_matches"] = (
    label_check["diagnosis_feature"] == label_check["diagnosis_patient"]
)

print("FEATURE LABEL CHECK")
print("Feature patient-label rows:", len(feature_labels))
print("Unmatched patient IDs:", (label_check["_merge"] != "both").sum())
print("Diagnosis mismatches:", (~label_check["label_matches"]).sum())

# 2. Check cycle table integrity.
print("\nCYCLE TABLE CHECK")
print("Cycle rows:", len(cycles))
print("Feature rows:", len(features))
print("Missing patient IDs in cycles:", cycles["patient_id"].isna().sum())
print("Missing recording IDs in cycles:", cycles["recording_id"].isna().sum())

if "cycle_number" in cycles.columns:
    cycle_key = [
        "patient_id", "recording_id",
        "chest_location", "cycle_number"
    ]
    print(
        "Duplicate cycle keys:",
        cycles.duplicated(cycle_key).sum()
    )

# 3. Check whether audio paths identify individual manifest rows.
print("\nMANIFEST AUDIO-PATH CHECK")
for name, df in splits.items():
    print(f"\n{name.upper()}")
    print("Rows:", len(df))
    print("Unique audio paths:", df["audio_path"].nunique())
    print("Duplicate audio paths:", df["audio_path"].duplicated().sum())

# 4. Check cross-split audio-path overlap.
path_sets = {
    name: set(df["audio_path"].dropna().astype(str))
    for name, df in splits.items()
}

print("\nCROSS-SPLIT AUDIO-PATH OVERLAP")
for first, second in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test"),
]:
    shared = path_sets[first] & path_sets[second]
    print(f"{first} vs {second}: {len(shared)} shared paths")

# 5. Check manifest label consistency at patient level.
print("\nPATIENT DIAGNOSIS CONSISTENCY")
for name, df in splits.items():
    per_patient = df.groupby("patient_id")["diagnosis"].nunique()
    print(
        f"{name}: patients with multiple diagnoses = "
        f"{(per_patient > 1).sum()}"
    )



FEATURE LABEL CHECK
Feature patient-label rows: 126
Unmatched patient IDs: 0
Diagnosis mismatches: 0

CYCLE TABLE CHECK
Cycle rows: 6898
Feature rows: 6898
Missing patient IDs in cycles: 0
Missing recording IDs in cycles: 0
Duplicate cycle keys: 0

MANIFEST AUDIO-PATH CHECK

TRAIN
Rows: 4423
Unique audio paths: 4423
Duplicate audio paths: 0

VALIDATION
Rows: 914
Unique audio paths: 914
Duplicate audio paths: 0

TEST
Rows: 1561
Unique audio paths: 1561
Duplicate audio paths: 0

CROSS-SPLIT AUDIO-PATH OVERLAP
train vs validation: 0 shared paths
train vs test: 0 shared paths
validation vs test: 0 shared paths

PATIENT DIAGNOSIS CONSISTENCY
train: patients with multiple diagnoses = 0
validation: patients with multiple diagnoses = 0
test: patients with multiple diagnoses = 0
